In [1]:
pip install shap

   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
   - -------------------------------------- 1.0/41.9 MB 5.0 MB/s eta 0:00:09
   - -------------------------------------- 1.8/41.9 MB 5.0 MB/s eta 0:00:08
   -- ------------------------------------- 2.9/41.9 MB 4.7 MB/s eta 0:00:09
   --- ------------------------------------ 3.7/41.9 MB 4.5 MB/s eta 0:00:09
   ---- ----------------------------------- 4.5/41.9 MB 4.5 MB/s eta 0:00:09
   ----- ---------------------------------- 5.8/41.9 MB 4.7 MB/s eta 0:00:08
   ------- -------------------------------- 7.3/41.9 MB 5.0 MB/s eta 0:00:07
   -------- ------------------------------- 8.7/41.9 MB 5.1 MB/s eta 0:00:07
   --------- ------------------------------ 10.0/41.9 MB 5.3 MB/s eta 0:00:07
   ---------- ----------------------------- 11.3/41.9 MB 5.3 MB/s eta 0:00:06
   ----------- ---------------------------- 12.3/41.9 MB 5.4 MB/s eta 0:00:06
   ------------- -------------------------- 13.6/41.9 MB 5.5 MB/s eta 0:00:06
  

In [4]:
pip install groq


  Using cached groq-1.5.0-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.5.0-py3-none-any.whl (143 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import shap
import joblib

# load existing trained model
model = joblib.load('models/fraud_model_xgb.pkl')

# TreeExplainer is specifically optimized for XGBoost tree ensembles
explainer = shap.TreeExplainer(model)


In [ ]:
def explain_prediction(transaction_df):
    """
    transaction_df: single-row DataFrame with the same features
    used in training (V1-V28, Amount_log, Hour, Amount_bin, etc.)
    """
    shap_values= explainer.shap_values(transaction_df)
    #pair each feature  with its shap contrib ution
    feature_contributions= dict(zip(transaction_df.columns,shap_values[0]))

    top_features = sorted(
        feature_contributions.items(),
        key=lambda x: abs(x[1]),
        reverse=True)[:5] #top 5  drivers
    return top_features 
        



In [8]:
import os
from groq import Groq
from dotenv import load_dotenv

# Remove miniconda SSL cert conflict if present
os.environ.pop('SSL_CERT_FILE', None)
os.environ.pop('SSL_CERT_DIR', None)

api_key = os.environ.get('GROQ_API_KEY', '')
groq_client = Groq(api_key=api_key) if api_key else None

def generate_explanation(top_features, prediction, probability):
    if not groq_client:
        return 'GROQ_API_KEY not set. Please set GROQ_API_KEY in your environment to generate AI explanation.'
    system_instruction = 'You are a fraud analyst assistant.'
    factors_text = chr(10).join(f'- {f}: {v:+.4f}' for f, v in top_features)
    user_content = f'''A model flagged a transaction as {prediction} with {probability:.1%} confidence.
The top contributing factors (SHAP values, positive = pushes toward fraud) are:
{factors_text}
Write a 2-3 sentence plain-English explanation for a non-technical reviewer.'''
    messages = [
        {'role': 'system', 'content': system_instruction},
        {'role': 'user', 'content': user_content}
    ]
    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=messages,
        max_tokens=150,
    )
    return response.choices[0].message.content.strip()


In [9]:
import joblib
import numpy as np
import pandas as pd

# Load feature names and test dataset
feature_names = joblib.load('models/feature_names.pkl')
X_test_arr = np.load('data/processed/X_test.npy')
y_test_arr = np.load('data/processed/y_test.npy')

# Find the first true fraud transaction in the test set
fraud_indices = np.where(y_test_arr == 1)[0]
sample_idx = fraud_indices[0] if len(fraud_indices) > 0 else 0

sample_transaction = pd.DataFrame([X_test_arr[sample_idx]], columns=feature_names)

top_feats = explain_prediction(sample_transaction)
prediction = 'FRAUD' if model.predict(sample_transaction.values)[0] == 1 else 'NORMAL'
probability = float(model.predict_proba(sample_transaction.values)[0][1])

print(f'=== Sample Transaction #{sample_idx} ===')
print(f'Prediction:  {prediction} ({probability:.2%} probability)')
print('\nTop 5 SHAP Drivers:')
for feat, val in top_feats:
    direction = 'pushes towards FRAUD' if val > 0 else 'pulls towards NORMAL'
    print(f'  {feat:15s}: {val:+.4f} ({direction})')

explanation = generate_explanation(top_feats, prediction, probability)
print('\n=== Forensic Summary ===')
print(explanation)


In [6]:
import os
print(os.environ.get("SSL_CERT_FILE"))

C:\Users\Hp\miniconda3\envs\frauddetection/ssl/cacert.pem


In [7]:
import os
print(os.path.exists(r"C:\Users\Hp\miniconda3\envs\frauddetection\ssl\cacert.pem"))

False
